# Experiment: Phase 1 Antigravity Run

Preflight -> Antigravity GUI guard -> DryRun -> Manual/Semi-Manual Full Run -> Aggregate -> Review


## 목적

이 노트북은 Phase 1 실험을 특정 agent/editor 하나에 맞춰 실행하기 위한 파일입니다. 한 노트북에서는 하나의 `agent + model + reasoning/mode` 조합만 실행하여 결과 해석이 섞이지 않게 합니다.

기본 실행 규모는 `5 조건 x 3 반복 x 3 task = 45 run`입니다. 실제 비용과 시간을 막기 위해 `RUN_FULL_RUN=False`가 기본값입니다.


## 00 Configuration


In [ ]:
from pathlib import Path
import datetime
import subprocess

AGENT = "antigravity"
MODEL = "gemini-3-flash"
REASONING_EFFORT = "default"  # codex only: default|minimal|low|medium|high|xhigh
CODEX_REASONING_FLAG = "--reasoning-effort"

CONDITIONS = "C0,C1,C2,C3,C4"
REPEATS = 3
TASKS = "task-1-todo-crud,task-2-jwt,task-3-pagination"
TIME_BUDGET_MINUTES = 360
FAILURE_THRESHOLD = 6  # runner에는 직접 전달되지 않으며 실행 규모 확인용

RUN_ID = f"phase1-antigravity-{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"
RUN_FULL_RUN = False  # 비용/시간 보호: 실제 풀런 전 True로 변경
RUN_LIVE_SMOKE = True  # 실제 agent 최소 호출. 토큰/크레딧을 소량 사용할 수 있음
INSTALL_REQUIREMENTS = True

AGENT_EXTRA_ARGS = "Mode=agent Launch=true"
ANTIGRAVITY_MODE = "agent"
CONFIRM_ANTIGRAVITY_MODEL_SELECTED = False  # antigravity full run 전에 UI 모델/모드 확인 후 True

VALID_REASONING_EFFORTS = {"default", "minimal", "low", "medium", "high", "xhigh"}
if REASONING_EFFORT not in VALID_REASONING_EFFORTS:
    raise ValueError(f"REASONING_EFFORT must be one of {sorted(VALID_REASONING_EFFORTS)}")
if REASONING_EFFORT != "default" and AGENT != "codex":
    raise ValueError("REASONING_EFFORT is currently wired only for AGENT='codex'. Use 'default' for this notebook.")


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "experiments" / "runner" / "run_experiment.ps1").exists():
            return candidate
    raise RuntimeError("repo root를 찾지 못했습니다. 노트북을 repo 안에서 실행하세요.")

REPO_ROOT = find_repo_root(Path.cwd())
RUNNER = REPO_ROOT / "experiments" / "runner" / "run_experiment.ps1"
OUTPUT_ROOT = REPO_ROOT / "experiments" / "results"
RUN_ROOT = OUTPUT_ROOT / RUN_ID


def safe_segment(value: str) -> str:
    table = str.maketrans({c: "_" for c in '\\/:*?"<>|'})
    return value.translate(table)

SAFE_MODEL = safe_segment(MODEL)
SAFE_REASONING = safe_segment(REASONING_EFFORT)
AGENT_ROOT = RUN_ROOT / AGENT / SAFE_MODEL
if REASONING_EFFORT != "default":
    AGENT_ROOT = AGENT_ROOT / f"reasoning-{SAFE_REASONING}"

matrix_count = len([x for x in CONDITIONS.split(',') if x.strip()]) * REPEATS * len([x for x in TASKS.split(',') if x.strip()])
print(f"repo root       : {REPO_ROOT}")
print(f"agent           : {AGENT}")
print(f"model           : {MODEL}")
print(f"reasoning       : {REASONING_EFFORT}")
print(f"run id          : {RUN_ID}")
print(f"matrix          : {matrix_count} run(s)")
print(f"agent root      : {AGENT_ROOT}")
print(f"run full        : {RUN_FULL_RUN}")
if AGENT_EXTRA_ARGS:
    print(f"agent extra args: {AGENT_EXTRA_ARGS}")


## Helper Functions


In [ ]:
class PreflightError(RuntimeError):
    pass

PREFLIGHT_OK = False
AGENT_GUARD_OK = False
DRYRUN_OK = False
FULLRUN_OK = False


def run_cmd(args, *, cwd=None, timeout=120, check=False, env=None):
    shell = isinstance(args, str)
    printable = args if shell else " ".join(str(a) for a in args)
    print(f"$ {printable}")
    try:
        result = subprocess.run(
            args,
            cwd=str(cwd or REPO_ROOT),
            text=True,
            encoding="utf-8",
            errors="replace",
            capture_output=True,
            timeout=timeout,
            shell=shell,
            env=env,
        )
    except subprocess.TimeoutExpired as exc:
        raise PreflightError(f"command timed out after {timeout}s: {printable}") from exc

    if result.stdout:
        print(result.stdout[-6000:])
    if result.stderr:
        print(result.stderr[-6000:])
    if check and result.returncode != 0:
        raise PreflightError(f"command failed with exit code {result.returncode}: {printable}")
    return result


def ps_runner_args(*items):
    return [
        "powershell",
        "-NoProfile",
        "-ExecutionPolicy",
        "Bypass",
        "-File",
        str(RUNNER),
        *[str(x) for x in items],
    ]


def base_runner_args():
    args = [
        "-RunId", RUN_ID,
        "-OutputRoot", str(OUTPUT_ROOT),
        "-Agent", AGENT,
        "-Model", MODEL,
        "-ReasoningEffort", REASONING_EFFORT,
        "-ReasoningFlag", CODEX_REASONING_FLAG,
        "-Conditions", CONDITIONS,
        "-Repeats", str(REPEATS),
        "-Tasks", TASKS,
        "-AdapterTimeoutSec", str(TIME_BUDGET_MINUTES * 60),
    ]
    if AGENT_EXTRA_ARGS:
        args += ["-AgentExtraArgs", AGENT_EXTRA_ARGS]
    return args


def require_guard_ready():
    if not PREFLIGHT_OK:
        raise PreflightError("01 Environment Preflight 셀을 먼저 성공시켜야 합니다.")
    if not AGENT_GUARD_OK:
        raise PreflightError("02 Agent/Model Guard 셀을 먼저 성공시켜야 합니다.")


def runs_csv_path():
    return AGENT_ROOT / "runs.csv"


## 01 Environment Preflight


In [ ]:
global PREFLIGHT_OK
if not RUNNER.exists():
    raise PreflightError(f"runner not found: {RUNNER}")

venv_python = REPO_ROOT / ".venv" / "Scripts" / "python.exe"
requirements = REPO_ROOT / "requirements.txt"
if not venv_python.exists():
    raise PreflightError(f".venv python not found: {venv_python}")
if not requirements.exists():
    raise PreflightError(f"requirements.txt not found: {requirements}")

version = run_cmd([str(venv_python), "--version"], check=True)
if INSTALL_REQUIREMENTS:
    run_cmd([str(venv_python), "-m", "pip", "install", "-r", str(requirements)], timeout=600, check=True)

PREFLIGHT_OK = True
print("Environment preflight OK")


## 02 Agent/Model Guard


In [ ]:
global AGENT_GUARD_OK
AGENT_GUARD_OK = False

version = run_cmd(["antigravity", "--version"], timeout=60, check=True)
chat_help = run_cmd(["antigravity", "chat", "--help"], timeout=60, check=True)
help_text = chat_help.stdout + chat_help.stderr
if "--mode" not in help_text:
    raise PreflightError("antigravity chat --help에서 --mode 옵션을 확인하지 못했습니다.")

print("Antigravity CLI guard OK")
print(f"Recorded model label: {MODEL}")
print(f"Recorded UI mode    : {ANTIGRAVITY_MODE}")
print("주의: Antigravity CLI는 현재 --model 옵션을 노출하지 않습니다.")
print("따라서 MODEL 값은 실험 기록 라벨이며, 실제 모델 선택은 Antigravity UI에서 직접 맞춰야 합니다.")

if RUN_LIVE_SMOKE:
    print("Antigravity는 headless model smoke를 수행하지 않습니다. GUI에서 모델 선택을 확인하세요.")

AGENT_GUARD_OK = True


## Antigravity Manual/Semi-Manual Flow

이 노트북은 `agent=antigravity`로 runner를 실행하지만, Antigravity CLI가 안정적인 `--model` 옵션을 제공하지 않으므로 실제 모델 선택은 UI에서 직접 수행합니다.

Full Run 셀을 실행하기 전 확인할 것:

- Antigravity에서 노트북 설정의 `MODEL`과 같은 모델을 선택합니다.
- Antigravity에서 `ANTIGRAVITY_MODE`와 같은 모드 또는 가장 가까운 UI 모드를 선택합니다.
- 설정 셀의 `CONFIRM_ANTIGRAVITY_MODEL_SELECTED = True`로 바꿉니다.
- 각 run마다 adapter가 workspace와 prompt 파일을 보여주면 Antigravity에서 작업을 수행하고 Enter를 눌러 평가로 넘깁니다.


## 03 DryRun


In [ ]:
global DRYRUN_OK
require_guard_ready()
cmd = ps_runner_args(*base_runner_args(), "-DryRun", "-NoEvaluate")
dry = run_cmd(cmd, timeout=600, check=True)
rcsv = runs_csv_path()
if not rcsv.exists():
    raise RuntimeError(f"DryRun runs.csv가 생성되지 않았습니다: {rcsv}")
DRYRUN_OK = True
print(f"DryRun OK: {rcsv}")


## Quota Monitor Command


In [ ]:
stop_file = RUN_ROOT / ".stop"
quota_cmd = [
    "powershell",
    "-NoProfile",
    "-ExecutionPolicy",
    "Bypass",
    "-File",
    str(REPO_ROOT / "experiments" / "runner" / "monitor_quota.ps1"),
    "-StopFile",
    str(stop_file),
]
print("Quota monitor는 백그라운드 프로세스를 남기지 않도록 자동 실행하지 않습니다.")
print("별도 PowerShell 터미널에서 필요할 때 아래 명령을 실행하세요:")
print(" ".join(quota_cmd))


## 04 Full Run


In [ ]:
global FULLRUN_OK
require_guard_ready()
if not DRYRUN_OK:
    raise PreflightError("03 DryRun 셀을 먼저 성공시켜야 합니다.")
if not RUN_FULL_RUN:
    print("Full run skipped: RUN_FULL_RUN=False 입니다. 비용/시간을 확인한 뒤 설정 셀에서 True로 바꾸세요.")
elif AGENT == "antigravity" and not CONFIRM_ANTIGRAVITY_MODEL_SELECTED:
    print("Antigravity full run skipped: CONFIRM_ANTIGRAVITY_MODEL_SELECTED=False 입니다.")
    print("UI에서 MODEL/Mode를 맞출 준비가 되면 설정 셀에서 True로 바꾸세요.")
else:
    cmd = ps_runner_args(*base_runner_args())
    result = run_cmd(cmd, timeout=TIME_BUDGET_MINUTES * 60 + 600, check=False)
    if result.returncode != 0:
        raise RuntimeError(f"Full run failed with exit code {result.returncode}")
    FULLRUN_OK = True
    print("Full run OK")


## 05 Aggregate


In [ ]:
if not RUN_ROOT.exists():
    raise RuntimeError(f"Run root not found: {RUN_ROOT}")
cmd = [
    str(REPO_ROOT / ".venv" / "Scripts" / "python.exe"),
    str(REPO_ROOT / "experiments" / "runner" / "aggregate.py"),
    "--run-root",
    str(RUN_ROOT),
]
agg = run_cmd(cmd, timeout=600, check=True)
print(f"Aggregate OK: {RUN_ROOT}")


## 06 Review Results


In [ ]:
import pandas as pd

summary_csv = RUN_ROOT / "summary.csv"
report_md = RUN_ROOT / "report.md"
print(f"summary.csv: {summary_csv}")
print(f"report.md  : {report_md}")
if not summary_csv.exists():
    raise RuntimeError("summary.csv가 아직 없습니다. Full Run과 Aggregate를 먼저 실행하세요.")

df = pd.read_csv(summary_csv)
display(df.head())
metric_cols = [
    c for c in [
        "build_success",
        "test_success",
        "design_alignment_score",
        "doc_traceability_score",
        "final_score",
        "wall_sec",
    ] if c in df.columns
]
group_cols = [c for c in ["agent", "model", "reasoning_effort", "condition"] if c in df.columns]
if group_cols and metric_cols:
    display(df.groupby(group_cols)[metric_cols].mean(numeric_only=True).reset_index())
else:
    print("pivot에 필요한 컬럼이 부족합니다.")
